# C2.4 · Data-layer research

**Function C — Red Teaming and Security Research with AI → Security Research with AI**  ·  *Security of AI*

Builds on **[C2.3 · Weight-level techniques](https://spbreed.github.io/cyber-commons/lessons/C2.3.html)**.

| | |
|---|---|
| Tools used | Qdrant, sentence-transformers |

## What this lesson is

**What it covers.** Invert embeddings from a local vector store and recover source text.

**Why a security engineer needs it.** Memorisation, extraction, embedding inversion, index poisoning. The control it builds is: measure extraction rates rather than assert privacy.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The corpus is a write surface. Training data, a RAG index and an agent's memory are three versions of the same problem: text somebody else authored, read back later as fact, long after anyone remembers where it came from.

> **At CyberTravels.** The vector store behind the RAG Advisor is a write surface. Anything ingested once is read back as fact long after anyone remembers where it came from. R12.

## 2 · The framework

```
   three names for one problem: text somebody else wrote, read back as fact

   training data --+
   RAG corpus    --+--> context window --> the agent believes it
   agent memory  --+

   detection differs per layer; provenance is the control in all three
```

Data-layer research has one governing result: **provenance beats volume.**

Published data-poisoning attacks succeed at contamination rates well under 1%,
and some at a few hundred documents regardless of corpus size. That breaks the
intuition most teams operate on — "we have a lot of clean data, a few bad
records will be drowned out". They will not.

If volume does not protect you, the only thing that does is knowing **exactly
what is in the corpus**: per-record hashes, a signed manifest, and the ability to
answer "which records changed since the snapshot we signed off?"

That capability also happens to be what a privacy erasure request needs, which
is why E2.5 depends on this lesson.

## 3 · Where it breaks — a corpus you cannot describe

The practical failure is not that poisoning is undetectable. It is that most teams cannot answer basic questions about the corpus that trained the model currently in production.

## 4 · The procedure, as a skill

The skill prints what 0.01%, 0.1% and 1% mean as a record count for a real corpus size, then builds the hashed manifest — because a record list answers none of the four questions and a manifest with a root answers all four.

### The skill — [`skills/research/training-data-provenance-manifest/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/training-data-provenance-manifest/SKILL.md)

```yaml
name: training-data-provenance-manifest
description: >-
  Compute what fraction of a corpus an attacker needs to poison it, and build a
  hashed manifest that can answer where a record came from and whether it
  changed. Use when reviewing training or fine-tuning data, or a RAG corpus
  nobody can attest to.
allowed-tools: Read, Grep, Glob
```

# A list of records is not provenance

Data-layer attacks need a smaller share of a corpus than people expect, so the
useful question is not "could someone poison this" but "could we tell". A record
list answers none of the four questions that matter; a manifest of content
hashes with a root answers all four, and it is cheap.

## When to use this

Any corpus that trains, fine-tunes or grounds a model — including the RAG index
somebody built from a shared drive.

## Procedure

**1 — State the poisoning rates in absolute terms.** For the corpus size you
have, print what 0.01%, 0.1% and 1% mean as a record count. The number is
usually small enough to end the argument about whether it is feasible.

**2 — Write down the four questions.** Where did this record come from, has it
changed since ingestion, what is in the corpus now, and what was in it at
training time. These are the requirements.

**3 — Compare what each artefact can answer.** A record list, a row count, a
snapshot, a hashed manifest. Only the last answers all four, and showing the
table is more persuasive than asserting it.

**4 — Build the manifest.** Content hash per record plus its source, and a root
over the whole set. The root is what makes "the corpus changed" a one-comparison
question.

**5 — Demonstrate detection.** Append records, recompute, and show both that the
root moved and which records are new. A manifest that detects change without
localising it sends you back to diffing the corpus.

## Output contract

```json
{
  "corpus": {"records": 0, "poison_rates": [{"rate": 0.0, "records": 0}]},
  "questions": [{"question": "str", "answerable_by": ["str"]}],
  "manifest": {"records": 0, "root": "str", "per_record": [{"id": "str", "hash": "str", "source": "str"}]},
  "detection": {"appended": 0, "root_changed": true, "localised": ["str"]}
}
```

## Failure modes

- **Arguing about feasibility.** Print the record count and the argument ends.
- **A manifest with no source field.** It answers "changed", never "from where".
- **A root with no per-record hashes.** Detection without localisation.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/training-data-provenance-manifest/scripts/training_data_provenance_manifest.py
SCRIPT = "skills/research/training-data-provenance-manifest/scripts/training_data_provenance_manifest.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Poison rates of 0.01%, 0.1% and 1% print for a 100,000-record corpus. The capability table shows only hashed manifests can answer the four questions. The manifest root changes when three records are appended and the three new records are identified by hash, including the injected one, which is then located and removed exactly.

## Your turn

For one dataset feeding a production model, try to produce the hash of the exact snapshot that trained the deployed version. Time-box it to an hour. The answer usually arrives in ten minutes and is usually no.

---

**Next → [C2.5 · Supply-chain research](https://spbreed.github.io/cyber-commons/lessons/C2.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*